<a href="https://colab.research.google.com/github/suhaani1/FitPulse-Health-Anomaly-Detection-from-Fitness-Devices/blob/main/Milestone_4_Dashboard/dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install fastapi uvicorn streamlit pyngrok pandas requests prophet tsfresh scikit-learn matplotlib    --quiet

In [12]:
from pyngrok import ngrok

ngrok.set_auth_token("33dxk8ZjuuGtV97ggXXTDWuaaaB_7QcrpYRewXcXr8u6UB7Xo")


In [13]:
%%writefile backend.py
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans

app = FastAPI(title="FitPulse Unified Backend")

# ======================================================
# GLOBAL STATE (COLAB SAFE)
# ======================================================
DATA = None
FEATURED_DATA = None
ANOMALIES = None

# ======================================================
# 1. PREPROCESSING
# ======================================================
def preprocess_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Standardize column names
    df.rename(columns={
        "Id": "user_id",
        "Heart_rate": "heart_rate",
        "Steps": "steps",
        "SleepValue": "sleep"
    }, inplace=True)

    if "date" not in df.columns:
        raise ValueError("Missing 'date' column")

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    df = df.sort_values(["user_id", "date"])

    if "heart_rate" in df.columns:
        df["heart_rate"] = df["heart_rate"].interpolate().bfill().ffill()

    if "steps" in df.columns:
        df["steps"] = df["steps"].fillna(0)

    if "sleep" in df.columns:
        df["sleep"] = df["sleep"].ffill().bfill()

    return df


# ======================================================
# 2. FEATURE EXTRACTION + MODELING
# ======================================================
def feature_engineering(df, window=7):
    df = df.copy()

    for metric in ["heart_rate", "sleep", "steps"]:
        if metric in df.columns:
            df[f"{metric}_trend"] = df[metric].rolling(
                window=window, center=True
            ).mean()

            df[f"{metric}_residual"] = (
                df[metric] - df[f"{metric}_trend"]
            )

    return df


# ======================================================
# 3. CLUSTERING (UNSUPERVISED BEHAVIOR GROUPING)
# ======================================================
def clustering(df):
    df = df.copy()

    features = []

    for col in ["heart_rate", "sleep", "steps"]:
        if col in df.columns:
            features.append(col)

    if len(features) < 2:
        df["cluster"] = 0
        return df

    X = df[features].fillna(0)

    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df["cluster"] = kmeans.fit_predict(X)

    return df


# ======================================================
# 4. RULE-BASED ANOMALY DETECTION
# ======================================================
def detect_anomalies(df, z_thresh=2.0):
    df = df.copy()
    df["anomaly"] = False

    if "heart_rate_residual" in df.columns:
        std = df["heart_rate_residual"].std()
        if std > 0:
            df["anomaly"] |= (
                np.abs(df["heart_rate_residual"]) > z_thresh * std
            )

    if "sleep" in df.columns:
        df["anomaly"] |= (df["sleep"] < 4)

    if "steps" in df.columns:
        df["anomaly"] |= (df["steps"] == 0)

    return df


# ======================================================
# 5. API ENDPOINTS
# ======================================================
@app.post("/upload")
async def upload_data(file: UploadFile = File(...)):
    global DATA, FEATURED_DATA, ANOMALIES

    try:
        df = pd.read_csv(file.file)
        df = preprocess_data(df)

        DATA = df
        df.to_csv("/content/clean_data.csv", index=False)

        return {"rows": len(df)}

    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})


@app.post("/analyze")
def analyze():
    global FEATURED_DATA, ANOMALIES

    if DATA is None:
        return JSONResponse(
            status_code=400,
            content={"error": "No data uploaded"}
        )

    df = feature_engineering(DATA)
    df = clustering(df)
    df = detect_anomalies(df)

    FEATURED_DATA = df
    ANOMALIES = df[df["anomaly"]]

    df.to_csv("/content/featured_data.csv", index=False)
    ANOMALIES.to_csv("/content/anomaly_report.csv", index=False)

    return {
        "total_records": len(df),
        "anomalies": len(ANOMALIES)
    }


@app.get("/anomalies")
def get_anomalies():
    if ANOMALIES is None:
        return []
    return ANOMALIES[
        ["user_id", "date", "heart_rate", "sleep", "steps", "cluster"]
    ].to_dict("records")


Overwriting backend.py


In [14]:
import threading
import uvicorn
import backend   # THIS IMPORT IS IMPORTANT

def run_backend():
    uvicorn.run(
        backend.app,   # 👈 pass the app object directly
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

threading.Thread(target=run_backend, daemon=True).start()

print("✅ FitPulse Backend running at http://localhost:8000")


✅ FitPulse Backend running at http://localhost:8000


INFO:     Started server process [624]


In [15]:
import requests

try:
    r = requests.get("http://localhost:8000/docs")
    print("Backend status:", r.status_code)
except Exception as e:
    print("Backend NOT running:", e)


INFO:     Application startup complete.


INFO:     127.0.0.1:46816 - "GET /docs HTTP/1.1" 200 OK
Backend status: 200


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [16]:
import requests
r = requests.post("http://localhost:8000/upload")
print(r.status_code)
print(r.text)


INFO:     127.0.0.1:46830 - "POST /upload HTTP/1.1" 422 Unprocessable Entity
422
{"detail":[{"type":"missing","loc":["body","file"],"msg":"Field required","input":null}]}


In [17]:
!mkdir -p ~/.streamlit


In [18]:
%%writefile ~/.streamlit/config.toml
[theme]
base = "light"
primaryColor = "#2A9D8F"
backgroundColor = "#EAF4F8"
secondaryBackgroundColor = "#D6ECF3"
textColor = "#0B3C5D"
font = "sans serif"


Overwriting /root/.streamlit/config.toml


In [33]:
%%writefile app.py
import streamlit as st
import pandas as pd
import requests
import plotly.express as px
import os

# =====================================================
# CONFIG
# =====================================================
BACKEND_URL = "http://localhost:8000"

st.set_page_config(
    page_title="FitPulse – Health Anomaly Detection",
    layout="wide"
)

# =====================================================
# MEDICAL UI THEME
# =====================================================
st.markdown("""
<style>
/* App background */
html, body, [class*="css"] {
    background-color: #F5F7FB !important;
}

/* Main app */
.stApp {
    background-color: #F5F7FB !important;
}

/* Sidebar */
section[data-testid="stSidebar"] {
    background-color: #EEF2FF !important;
}

/* Metric cards */
div[data-testid="metric-container"] {
    background-color: #FFFFFF !important;
    padding: 18px !important;
    border-radius: 14px !important;
    box-shadow: 0 6px 16px rgba(99,102,241,0.15) !important;
}

/* Buttons */
button[kind="primary"] {
    background-color: #4F46E5 !important;  /* Indigo */
    color: white !important;
    border-radius: 10px !important;
}

button {
    border-radius: 10px !important;
}

/* Headings */
h1 {
    color: #312E81 !important;
}

h2, h3 {
    color: #4338CA !important;
}

/* Text */
p, label, span {
    color: #374151 !important;
}
</style>
""", unsafe_allow_html=True)


# =====================================================
# HEADER
# =====================================================
st.markdown("""
<h1>FitPulse – Health Anomaly Detection System</h1>
<p style="color:#4B5563;">
An integrated platform for health data preprocessing, feature extraction,
clustering, and anomaly detection
</p>
""", unsafe_allow_html=True)


st.divider()

# =====================================================
# SIDEBAR – DATA UPLOAD
# =====================================================
st.sidebar.header(" Upload Fitness Data")

uploaded_file = st.sidebar.file_uploader(
    "Upload CSV file",
    type=["csv"]
)

if uploaded_file:
    with st.spinner("Uploading & preprocessing data..."):
        res = requests.post(
            f"{BACKEND_URL}/upload",
            files={"file": uploaded_file}
        )

    if res.status_code == 200:
        st.sidebar.success("Data uploaded successfully")
        st.sidebar.metric("Records Loaded", res.json()["rows"])
    else:
        st.sidebar.error(res.json().get("error", "Upload failed"))

# =====================================================
# LOAD CLEAN DATA (FROM BACKEND OUTPUT)
# =====================================================
CLEAN_PATH = "/content/clean_data.csv"

if not os.path.exists(CLEAN_PATH):
    st.info("⬅ Upload a CSV file to activate the dashboard")
    st.stop()

df = pd.read_csv(CLEAN_PATH, parse_dates=["date"])

# =====================================================
# GLOBAL FILTERS
# =====================================================
st.subheader(" Filters")

c1, c2, c3 = st.columns(3)

with c1:
    users = ["All"] + sorted(df["user_id"].astype(str).unique().tolist())
    selected_user = st.selectbox("User", users)

with c2:
    start_date, end_date = st.date_input(
        "Date Range",
        value=[df["date"].min(), df["date"].max()]
    )

with c3:
    metric = st.selectbox(
        "Metric",
        ["heart_rate", "sleep", "steps"]
    )

if selected_user != "All":
    df = df[df["user_id"].astype(str) == selected_user]

df = df[
    (df["date"] >= pd.to_datetime(start_date)) &
    (df["date"] <= pd.to_datetime(end_date))
]

st.divider()

# =====================================================
# TABS
# =====================================================
tab1, tab2, tab3, tab4, tab5 = st.tabs([
    " Overview",
    " Modeling & Trends",
    " Clustering",
    " Anomalies",
    " Reports"
])

# =====================================================
# TAB 1 – OVERVIEW
# =====================================================
with tab1:
    st.subheader("Dataset Overview")

    c1, c2, c3 = st.columns(3)
    c1.metric("Users", df["user_id"].nunique())
    c2.metric("Days", df["date"].nunique())
    c3.metric("Records", len(df))

    fig = px.line(
        df,
        x="date",
        y=metric,
        color="user_id",
        template="plotly_white",
        title=f"{metric.replace('_',' ').title()} Trend"
    )
    st.plotly_chart(fig, use_container_width=True)

    st.dataframe(df.head(50), use_container_width=True)

# =====================================================
# TAB 2 – MODELING & FEATURE ENGINEERING
# =====================================================
with tab2:
    st.subheader("Trend Modeling & Residuals")

    if st.button("▶ Run Modeling Pipeline"):
        with st.spinner("Running feature engineering & modeling..."):
            res = requests.post(f"{BACKEND_URL}/analyze")

        if res.status_code == 200:
            st.success("Modeling completed")
            st.json(res.json())

    FEATURED_PATH = "/content/featured_data.csv"
    if os.path.exists(FEATURED_PATH):
        fdf = pd.read_csv(FEATURED_PATH, parse_dates=["date"])

        trend_col = f"{metric}_trend"
        if trend_col in fdf.columns:
            fig = px.line(
                fdf,
                x="date",
                y=[metric, trend_col],
                color="user_id",
                template="plotly_white",
                title="Observed vs Trend"
            )
            st.plotly_chart(fig, use_container_width=True)

# =====================================================
# TAB 3 – CLUSTERING
# =====================================================
with tab3:
    st.subheader("Behavioral Clustering")

    if os.path.exists(FEATURED_PATH):
        fdf = pd.read_csv(FEATURED_PATH)

        numeric_cols = [
            col for col in ["heart_rate", "steps", "sleep"]
            if col in fdf.columns
        ]

        if "cluster" not in fdf.columns:
            st.info("Run modeling pipeline to generate clusters")
            st.stop()

        if len(numeric_cols) < 2:
            st.warning("Not enough numeric features for clustering view")
            st.stop()

        c1, c2 = st.columns(2)
        with c1:
            x_axis = st.selectbox("X-axis", numeric_cols, index=0)
        with c2:
            y_axis = st.selectbox(
                "Y-axis",
                [c for c in numeric_cols if c != x_axis],
                index=0
            )
        # Force cluster to string
        fdf["cluster"] = fdf["cluster"].astype(str)

        cluster_colors = {
          0: "#3B82F6",  # Blue
           1: "#10B981",  # Green
           2: "#EF4444"   # Red
        }

        fig = px.scatter(
            fdf,
            x=x_axis,
            y=y_axis,
            color="cluster",
            template="plotly_white",
            title=f"Behavior Clusters: {x_axis} vs {y_axis}"
        )
        st.plotly_chart(fig, use_container_width=True)


# =====================================================
# TAB 4 – ANOMALIES
# =====================================================
with tab4:
    st.subheader("Detected Anomalies")

    res = requests.get(f"{BACKEND_URL}/anomalies")
    anomalies = pd.DataFrame(res.json())

    if not anomalies.empty:
        st.dataframe(anomalies, use_container_width=True)

        fig = px.scatter(
            anomalies,
            x="date",
            y="heart_rate",
            color="cluster",
            template="plotly_white",
            title="Heart Rate Anomalies"
        )
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("No anomalies detected yet")

# =====================================================
# TAB 5 – REPORTS
# =====================================================
with tab5:
    st.subheader("Downloads")

    if os.path.exists("/content/anomaly_report.csv"):
        st.download_button(
            "⬇ Download Anomaly Report",
            open("/content/anomaly_report.csv", "rb"),
            file_name="anomaly_report.csv"
        )

    if os.path.exists("/content/featured_data.csv"):
        st.download_button(
            "⬇ Download Featured Dataset",
            open("/content/featured_data.csv", "rb"),
            file_name="featured_data.csv"
        )

    st.download_button(
        "⬇ Download Clean Data",
        open("/content/clean_data.csv", "rb"),
        file_name="clean_data.csv"
    )


Overwriting app.py


In [20]:
# from pyngrok import ngrok

ngrok.kill()          # stop old tunnels
public_url = ngrok.connect(8501)
print(public_url)

!streamlit run app.py &>/content/logs.txt &


NgrokTunnel: "https://unvinous-marion-unvociferously.ngrok-free.dev" -> "http://localhost:8501"


In [27]:
from pyngrok import ngrok
ngrok.kill()

!streamlit run app.py &>/content/logs.txt &
public_url = ngrok.connect(8501)
print("Dashboard URL:", public_url)


Dashboard URL: NgrokTunnel: "https://unvinous-marion-unvociferously.ngrok-free.dev" -> "http://localhost:8501"
